## 0. Setup

In [ ]:
import os
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# ML: features
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer

# ML: algorithms
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans

# ML: pipeline & tuning
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    BinaryClassificationEvaluator,
    ClusteringEvaluator,
    MulticlassClassificationEvaluator,
)

# ML: statistics
from pyspark.ml.stat import Correlation, ChiSquareTest, Summarizer
from pyspark.ml.linalg import Vectors

print("All imports OK")

In [ ]:
spark = (
    SparkSession.builder
    .appName("SparkML_Taxi_Demo")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.defaultParallelism} workers")

# Path to taxi data
DATA_DIR  = "/home/alka/MLOPs-Class/spark_demo"
DATA_2023 = os.path.join(DATA_DIR, "yellow_tripdata_2023-01.parquet")
MODEL_DIR = "/tmp/sparkml_demo_model"

---
## 1. Load & Prepare Data

We load the Jan 2023 taxi data, engineer features, and cache a clean sample for all ML tasks.

In [ ]:
raw = spark.read.parquet(DATA_2023)
print(f"Raw rows: {raw.count():,}")
raw.printSchema()

In [ ]:
# Feature engineering: add derived columns
df = (
    raw
    .withColumn(
        "duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .withColumn("hour_of_day",  F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week",  F.dayofweek("tpep_pickup_datetime"))
    # Clean: sensible ranges only
    .filter(
        (F.col("fare_amount")    >  0)   & (F.col("fare_amount")    < 200) &
        (F.col("trip_distance")  >  0)   & (F.col("trip_distance")  <  50) &
        (F.col("duration_min")   >  0)   & (F.col("duration_min")   < 120) &
        (F.col("tip_amount")     >= 0)   & (F.col("tip_amount")     <  60) &
        (F.col("passenger_count").isNotNull())
    )
    .select(
        "VendorID", "passenger_count", "trip_distance",
        "PULocationID", "DOLocationID", "payment_type",
        "fare_amount", "tip_amount", "total_amount",
        "duration_min", "hour_of_day", "day_of_week",
    )
)

# Sample 5% rows for speed up
df_sample = df.sample(fraction=0.05, seed=42).cache()

n = df_sample.count()
print(f"Clean sample: {n:,} rows")
df_sample.show(5)

---
## 3. ML Pipeline Concepts  

The five main abstractions:

| Concept | Role | Taxi Example |
|---------|------|--------------|
| **DataFrame** | ML dataset | taxi trips with columns for distance, fare, features, label |
| **Transformer** | `DataFrame → DataFrame` | `VectorAssembler` appends a `features` column |
| **Estimator** | `fit(DataFrame) → Transformer` | `StandardScaler.fit(df)` learns means/stds → `StandardScalerModel` |
| **Pipeline** | chains Transformers + Estimators | `[Assembler, Scaler, LR]` |
| **Parameter** | uniform API | `lr.setMaxIter(10)` or `ParamMap(lr.maxIter→10)` |

### 3.1 DataFrame — the ML dataset

In [ ]:
# The DataFrame holds raw features, engineered features, labels, and predictions side-by-side
print("DataFrame columns:", df_sample.columns)
print(f"Rows: {df_sample.count():,}")
df_sample.select("trip_distance", "fare_amount", "tip_amount", "payment_type").show(5)

### 3.2 Transformer — `VectorAssembler`

A Transformer has a **`transform(df)`** method.  
It **appends** one or more columns — it never modifies existing columns.  
Each instance has a **unique UID**.

In [ ]:
feature_cols = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Each Transformer has a unique id
print(f"Transformer uid  : {assembler.uid}")
print(f"Type             : {type(assembler).__name__}")

# transform() appends the 'features' column — returns a NEW DataFrame
df_assembled = assembler.transform(df_sample)
print(f"\nColumns before: {len(df_sample.columns)}")
print(f"Columns after : {len(df_assembled.columns)}  (added: 'features')")
df_assembled.select("trip_distance", "fare_amount", "features").show(3, truncate=60)

### 3.3 Estimator — `StandardScaler`

An Estimator has a **`fit(df)`** method that **learns** from the data  
and returns a trained **Transformer** (a `StandardScalerModel`).  

> `Transformer.transform()` and `Estimator.fit()` are **stateless** — the  
> input DataFrame is never modified; a new one is returned.

In [ ]:
scaler = StandardScaler(
    inputCol="features", outputCol="scaled_features",
    withMean=True, withStd=True
)

print(f"Estimator uid    : {scaler.uid}")
print(f"Type             : {type(scaler).__name__}  ← Estimator (not yet trained)")

# fit() learns mean/std from the TRAINING data → produces a StandardScalerModel
scaler_model = scaler.fit(df_assembled)
print(f"\nAfter fit()")
print(f"Model type       : {type(scaler_model).__name__}  ← Transformer (trained)")
print(f"Learned means    : {scaler_model.mean.toArray().round(3)}")

# The model (Transformer) can now transform new data
df_scaled = scaler_model.transform(df_assembled)
df_scaled.select("features", "scaled_features").show(2, truncate=70)

### 3.4 Pipeline — chaining stages

A `Pipeline` is itself an **Estimator**.  
`pipeline.fit(df)` calls each stage in order:  
- If a stage is an **Estimator**, `fit()` is called and the resulting model is used for `transform()`  
- If a stage is a **Transformer**, `transform()` is called directly

In [ ]:
# Build the demo Assembler -> StandardScaler pipeline (df_out) and persist its
# scaled features + label to Parquet for the framework comparison
from pyspark.ml.functions import vector_to_array

feature_cols = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week"]

demo_assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
demo_scaler    = StandardScaler(inputCol="raw_features", outputCol="scaled_features",
                                 withMean=True, withStd=True)
demo_pipeline  = Pipeline(stages=[demo_assembler, demo_scaler])
demo_model     = demo_pipeline.fit(df_sample)
df_out         = demo_model.transform(df_sample)

reg_features = feature_cols
label_col    = "tip_amount"

df_cc = df_out.filter(F.col("payment_type") == 1)

PARQUET_PATH = "/home/alka/MLOPs-Class/ML_Framework/taxi_tip_regression.parquet"
if os.path.exists(PARQUET_PATH):
    shutil.rmtree(PARQUET_PATH)

(
    df_cc
    .select(
        *[vector_to_array("scaled_features")[i].alias(c) for i, c in enumerate(reg_features)],
        label_col,
    )
    .write.mode("overwrite").parquet(PARQUET_PATH)
)

print(f"Saved {df_cc.count():,} rows to: {PARQUET_PATH}")


---
## 5. Classification Pipeline — High Tip Prediction  

**Task:** Predict whether a credit-card trip will have a **high tip** (tip > 20% of fare).

```
df  →  VectorAssembler  →  LogisticRegression  →  predictions (label: 0/1)
```

In [ ]:
# Create binary label: high_tip = 1 if tip > 20% of fare
df_clf = df_cc.withColumn(
    "high_tip",
    (F.col("tip_amount") / F.col("fare_amount") > 0.20).cast("double")
)

print("Label distribution:")
df_clf.groupBy("high_tip").count().orderBy("high_tip").show()

train_clf, test_clf = df_clf.randomSplit([0.8, 0.2], seed=42)

In [ ]:
clf_features = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week"]

clf_assembler = VectorAssembler(inputCols=clf_features, outputCol="features")

logreg = LogisticRegression(
    featuresCol="features", labelCol="high_tip",
    maxIter=20, regParam=0.01
)

clf_pipeline = Pipeline(stages=[clf_assembler, logreg])

clf_model = clf_pipeline.fit(train_clf)
predictions_clf = clf_model.transform(test_clf)

# Evaluate
auc = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
).evaluate(predictions_clf)

acc = MulticlassClassificationEvaluator(
    labelCol="high_tip", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions_clf)

print(f"AUC-ROC  : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")

predictions_clf.select("high_tip", "prediction", "probability").show(6)

---
## 7. Parameters & Model Tuning 

### 7.1 The Parameters API

> MLlib uses a **uniform parameter API**: parameters can be set directly on an instance  
> (`lr.setMaxIter(10)`) or passed via a `ParamMap` to `fit()`/`transform()`.

In [ ]:
from pyspark.ml.param import Param

lr1 = LogisticRegression(featuresCol="features", labelCol="high_tip")
lr2 = LogisticRegression(featuresCol="features", labelCol="high_tip")

# Method 1: set directly on instance
lr1.setMaxIter(1)
lr1.setRegParam(0.01)
print("lr1 maxIter:", lr1.getMaxIter())

lr2.setMaxIter(20)
print("lr2 maxIter:", lr2.getMaxIter())


# Build a simple pipeline with two LogisticRegression stages (lr1 and lr2)
pipeline = Pipeline(stages=[clf_assembler, lr1])
model = pipeline.fit(train_clf)
predictions = model.transform(test_clf)
predictions.select("high_tip", "prediction", "probability").show(3)
# Evaluate
auc = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
).evaluate(predictions)

acc = MulticlassClassificationEvaluator(
    labelCol="high_tip", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)

print(f"AUC-ROC  : {auc:.4f}")
print(f"Accuracy : {acc:.4f}")

pipeline2= Pipeline(stages=[clf_assembler, lr2])
model2= pipeline2.fit(train_clf)
predictions2 = model2.transform(test_clf)
predictions2.select("high_tip", "prediction", "probability").show(3)
# Evaluate
auc2 = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
).evaluate(predictions2)

acc2 = MulticlassClassificationEvaluator(
    labelCol="high_tip", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions2)

print(f"AUC-ROC  : {auc2:.4f}")
print(f"Accuracy : {acc2:.4f}")

### 7.2 Cross Validator 

Equivalent to **GridSearchCV** in scikit-learn:
1. Split data into **k folds**
2. For each parameter combination: train on k-1 folds, validate on remaining fold
3. Compute **average metric** across folds
4. Choose **best parameters** and retrain on full dataset

Use `cv.setParallelism(n)` to evaluate multiple parameter combos concurrently.

In [ ]:
cv_assembler = VectorAssembler(inputCols=clf_features, outputCol="features")
cv_lr        = LogisticRegression(featuresCol="features", labelCol="high_tip")
cv_pipeline  = Pipeline(stages=[cv_assembler, cv_lr])

# ParamGridBuilder: defines the hyperparameter search grid
param_grid = (
    ParamGridBuilder()
    .addGrid(cv_lr.maxIter,  [10, 30])       # 2 values
    .addGrid(cv_lr.regParam, [0.01, 0.1])    # 2 values  → 4 combos total
    .build()
)

print(f"Total parameter combinations: {len(param_grid)}")
for i, pg in enumerate(param_grid):
    print(f"  Combo {i}: maxIter={pg[cv_lr.maxIter]}  regParam={pg[cv_lr.regParam]}")

In [ ]:
cv_evaluator = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator    = cv_pipeline,
    estimatorParamMaps = param_grid,
    evaluator    = cv_evaluator,
    numFolds     = 3,           # 3-fold CV
    parallelism  = 4,           # evaluate up to 4 combos concurrently
    seed         = 42,
)

print(f"Running {len(param_grid)}-combo × {3}-fold CV (= {len(param_grid)*3} model trains)...")
cv_model = cv.fit(train_clf)
print("Done.")

# Best model AUC on validation
best_auc_cv = max(cv_model.avgMetrics)
best_idx    = cv_model.avgMetrics.index(best_auc_cv)
best_params = param_grid[best_idx]

print(f"\nCV Average AUC per combo:")
for i, (pg, metric) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    marker = " ← best" if i == best_idx else ""
    print(f"  maxIter={pg[cv_lr.maxIter]:>3}  regParam={pg[cv_lr.regParam]:.3f}  AUC={metric:.4f}{marker}")

# Test set performance of best model
best_preds = cv_model.transform(test_clf)
best_auc_test = cv_evaluator.evaluate(best_preds)
print(f"\nBest model Test AUC: {best_auc_test:.4f}")

---
## 8. ML Persistence — Save & Load Pipeline 

In [ ]:
import shutil

MODEL_PATH = "/home/alka/MLOPs-Class/ML_Framework/sparkml_classification_pipeline"

# Clean previous save if exists
if os.path.exists(MODEL_PATH):
    shutil.rmtree(MODEL_PATH)

# CrossValidator already retrained the best PipelineModel on the full training
# data — save that PipelineModel (not the CrossValidatorModel wrapper)
best_pipeline_model = cv_model.bestModel
best_pipeline_model.save(MODEL_PATH)
print(f"Pipeline saved to: {MODEL_PATH}")

# Show what was saved on disk
for root, dirs, files in os.walk(MODEL_PATH):
    level = root.replace(MODEL_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:4]:
            print(f"{indent}  {f}")

In [ ]:
from pyspark.ml import PipelineModel

# Load the pipeline back from disk
loaded_model = PipelineModel.load(MODEL_PATH)
print(f"Loaded model type: {type(loaded_model).__name__}")
print("Loaded stages:")
for i, stage in enumerate(loaded_model.stages):
    print(f"  [{i}] {type(stage).__name__}")

# Verify predictions match the original model
loaded_preds = loaded_model.transform(test_clf)

auc_loaded = BinaryClassificationEvaluator(
    labelCol="high_tip", metricName="areaUnderROC"
).evaluate(loaded_preds)

print(f"\nOriginal model Test AUC : {best_auc_test:.4f}")
print(f"Loaded model Test AUC   : {auc_loaded:.4f}")


In [ ]:
spark.stop()
print("SparkSession stopped.")